In [8]:
import gym
import numpy as np
np.bool8 = bool
import time

#середовище мінного поля
custom_map = ['SFFF', 'FHFH', 'FFFH', 'HFFG']
env = gym.make('FrozenLake-v1', desc=custom_map, is_slippery=False)

#оцінка стратегії
def compute_value_function(env, policy, gamma= 0.9, theta=1e-5):
    value_table = np.zeros(env.observation_space.n)
    for _ in range(100):
        delta = 0
        for state in range(env.observation_space.n):
            action = policy[state]
            value = 0
            for prob, next_state, reward, done in env.P[state][action]:
                temp = reward + gamma * value_table[next_state]
                value = value + prob * temp
            diff = abs(value - value_table[state])
            if diff > delta:
                delta = diff
            value_table[state] = value
        if delta < theta:
            break
    return value_table

#покращення стратегії
def policy_iteration(env, gamma= 0.9):
    policy = np.zeros(env.observation_space.n)
    for _ in range(100):
        value_function = compute_value_function(env, policy, gamma)
        policy_stable = True
        for state in range(env.observation_space.n):
            old_action = policy[state]
            action_values = np.zeros(env.action_space.n)
            for action in range(env.action_space.n):
                total = 0
                for prob, next_state, reward, done in env.P[state][action]:
                    res = reward + gamma * value_function[next_state]
                    total = total + prob * res
                action_values[action] = total
            best_action = np.argmax(action_values)
            policy[state] = best_action
            if old_action != best_action:
                policy_stable = False
        if policy_stable == True:
            break
    return policy.astype(int), value_function

SOLDIER = '😎'

#візуалізація поля
def render_field(env, state):
    grid = env.unwrapped.desc.astype(str)
    grid = np.array(grid)
    size = grid.shape[0]
    row = state // size
    col = state % size
    visual = grid.copy()
    visual[visual == 'S'] = '.'
    visual[visual == 'F'] = '.'
    visual[visual == 'H'] = '💥'
    visual[visual == 'G'] = '🎯'
    visual[row][col] = SOLDIER
    for r in visual:
        line = ''
        for c in r:
            line = line + c + ' '
        print(line)
    print()

#старт місії
def show_render(env, policy):
    state = env.reset()
    done = False
    print('🎖 Початок місії:\n')

    while done == False:
        render_field(env, state)
        time.sleep(0.5)
        action = policy[state]
        action = int(action)
        print('Рух...')
        state, reward, done, info = env.step(action)
        if done == True:
            done = True
    render_field(env, state)
    if reward == 1:
        print('🔥 Ціль досягнута!!!')
    else:
        print('💀 Підрив на міні(((')

policy, value_function = policy_iteration(env)

print('Оптимальна стратегія:')
print(policy.reshape((4, 4)))
print('----------')

show_render(env, policy)

Оптимальна стратегія:
[[1 2 1 0]
 [1 0 1 0]
 [2 1 1 0]
 [0 2 2 0]]
----------
🎖 Початок місії:

😎 . . . 
. 💥 . 💥 
. . . 💥 
💥 . . 🎯 

Рух...
. . . . 
😎 💥 . 💥 
. . . 💥 
💥 . . 🎯 

Рух...
. . . . 
. 💥 . 💥 
😎 . . 💥 
💥 . . 🎯 

Рух...
. . . . 
. 💥 . 💥 
. 😎 . 💥 
💥 . . 🎯 

Рух...
. . . . 
. 💥 . 💥 
. . . 💥 
💥 😎 . 🎯 

Рух...
. . . . 
. 💥 . 💥 
. . . 💥 
💥 . 😎 🎯 

Рух...
. . . . 
. 💥 . 💥 
. . . 💥 
💥 . . 😎 

🔥 Ціль досягнута!!!
